In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score
)

In [2]:
df = pd.read_csv("parkinsons.data")

In [3]:
df = df.drop(columns=["name"])

In [4]:
X = df.drop(columns=["status"])
y = df["status"]

In [5]:
FEATURE_NAMES = list(X.columns)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [7]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)


In [9]:
print("=" * 50)
print("SVM RESULTS")
print("=" * 50)
 
param_grid = {
    "C":     [0.1, 1, 10, 100],
    "gamma": ["scale", "auto", 0.01, 0.001],
    "kernel": ["rbf"]
}
svm_grid = GridSearchCV(SVC(probability=True, random_state=42),
                        param_grid, cv=5, scoring="accuracy", n_jobs=-1)
svm_grid.fit(X_train_sc, y_train)
svm_best = svm_grid.best_estimator_
 
print(f"Best params: {svm_grid.best_params_}")
y_pred_svm = svm_best.predict(X_test_sc)
print(f"Test Accuracy : {accuracy_score(y_test, y_pred_svm):.4f}")
print(f"ROC-AUC       : {roc_auc_score(y_test, svm_best.predict_proba(X_test_sc)[:,1]):.4f}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred_svm)}")

SVM RESULTS
Best params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Test Accuracy : 0.8974
ROC-AUC       : 0.9621

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.80      0.80        10
           1       0.93      0.93      0.93        29

    accuracy                           0.90        39
   macro avg       0.87      0.87      0.87        39
weighted avg       0.90      0.90      0.90        39



In [10]:
cv_svm = cross_val_score(svm_best, X_train_sc, y_train, cv=5, scoring="accuracy")
print(f"5-Fold CV Mean: {cv_svm.mean():.4f} ± {cv_svm.std():.4f}")

5-Fold CV Mean: 0.9296 ± 0.0235


In [11]:
print("\n" + "=" * 50)
print("RANDOM FOREST RESULTS")
print("=" * 50)
 
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)   # RF doesn't need scaling
y_pred_rf = rf.predict(X_test)
print(f"Test Accuracy : {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"ROC-AUC       : {roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]):.4f}")


RANDOM FOREST RESULTS
Test Accuracy : 0.9231
ROC-AUC       : 0.9621


In [12]:
feat_imp = pd.Series(rf.feature_importances_, index=FEATURE_NAMES)
print(f"\nTop 5 important features:\n{feat_imp.sort_values(ascending=False).head(5)}")


Top 5 important features:
PPE            0.150716
spread1        0.109254
MDVP:Fo(Hz)    0.060157
Jitter:DDP     0.057883
NHR            0.056363
dtype: float64


In [13]:
os.makedirs("model", exist_ok=True)
joblib.dump(svm_best,      "model/parkinsons_model.pkl")
joblib.dump(scaler,        "model/scaler.pkl")
joblib.dump(FEATURE_NAMES, "model/feature_names.pkl")

['model/feature_names.pkl']

In [14]:
print("\n" + "=" * 50)
print("FILES SAVED")
print("=" * 50)
print("  model/parkinsons_model.pkl")
print("  model/scaler.pkl")
print("  model/feature_names.pkl")
print("\nRun this once locally, then commit the model/ folder to GitHub.")


FILES SAVED
  model/parkinsons_model.pkl
  model/scaler.pkl
  model/feature_names.pkl

Run this once locally, then commit the model/ folder to GitHub.
